# Orquestador de entrenamiento, monitorización de deriva y predicción

Este cuaderno ejecuta el flujo principal de entrenamiento y evaluación, análisis de deriva (PSI) y generación de predicciones sobre el conjunto ciego. Incluye además una etapa opcional de prueba de reentrenamiento con datos sintéticos etiquetados.

El análisis PSI es una señal de monitorización: **no demuestra por sí solo una degradación del modelo ni bloquea automáticamente la predicción**. Los datos sintéticos se usan para comprobar técnicamente el flujo de drift y comparación de modelos; no representan evidencia de rendimiento en producción.

## 1. Identificar la raíz del proyecto

Abre este cuaderno desde la raíz del repositorio o desde una subcarpeta del proyecto. El código busca automáticamente la carpeta que contiene los scripts de entrenamiento, monitorización y predicción.

In [1]:
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        required_files = (
            candidate / "src" / "train.py",
            candidate / "src" / "drift_monitoring.py",
            candidate / "src" / "predict.py",
            candidate / "src" / "retrain.py",
        )
        if all(path.is_file() for path in required_files):
            return candidate
    raise FileNotFoundError(
        "No se encontró la raíz del proyecto. Comprueba que el cuaderno esté dentro de "
        "la carpeta indicada y que existan src/train.py, "
        "src/drift_monitoring.py, src/predict.py y src/retrain.py."
    )

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
TRAIN_SCRIPT = PROJECT_ROOT / "src" / "train.py"
DRIFT_SCRIPT = PROJECT_ROOT / "src" / "drift_monitoring.py"
PREDICT_SCRIPT = PROJECT_ROOT / "src" / "predict.py"
RETRAIN_SCRIPT = PROJECT_ROOT / "src" / "retrain.py"
NEW_LABELED_PATH = PROJECT_ROOT / "data" / "new_labeled_data.csv"
RETRAIN_EVALUATION_PATH = PROJECT_ROOT / "data" / "retraining_evaluation.csv"
RETRAIN_DRIFT_REPORT_PATH = PROJECT_ROOT / "reports" / "retraining_drift_psi.csv"
RETRAIN_COMPARISON_PATH = PROJECT_ROOT / "reports" / "retraining_comparison.csv"
PREDICTIONS_PATH = PROJECT_ROOT / "outputs" / "blind_test_predictions.csv"
DRIFT_REPORT_PATH = PROJECT_ROOT / "reports" / "data_drift_psi.csv"

print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Intérprete de Python: {sys.executable}")
print(f"Script de entrenamiento: {TRAIN_SCRIPT}")
print(f"Script de monitorización: {DRIFT_SCRIPT}")
print(f"Script de predicción: {PREDICT_SCRIPT}")

Raíz del proyecto: C:\Users\angry\source\repos\repos_otros\6.Wizeline_test\DataScientist_wizeline
Intérprete de Python: c:\Users\angry\source\repos\repos_otros\6.Wizeline_test\DataScientist_wizeline\.venv\Scripts\python.exe
Script de entrenamiento: C:\Users\angry\source\repos\repos_otros\6.Wizeline_test\DataScientist_wizeline\src\train.py
Script de monitorización: C:\Users\angry\source\repos\repos_otros\6.Wizeline_test\DataScientist_wizeline\src\drift_monitoring.py
Script de predicción: C:\Users\angry\source\repos\repos_otros\6.Wizeline_test\DataScientist_wizeline\src\predict.py


## 2. Función para ejecutar scripts y mostrar sus mensajes

La salida estándar y los errores se muestran en tiempo real. Los procesos utilizan el mismo intérprete de Python del kernel. Selecciona el kernel del entorno `.venv` antes de ejecutar el cuaderno.

Para evitar problemas de acentuación en Windows, los procesos hijos fuerzan UTF-8 mediante `PYTHONUTF8` y `PYTHONIOENCODING`.

In [2]:
import subprocess
import os

def run_script(
    script_path: Path,
    stage_name: str,
    args: list[str] | None = None,
) -> None:
    if not script_path.is_file():
        raise FileNotFoundError(f"No se encontró el script: {script_path}")

    print("=" * 72, flush=True)
    print(f"INICIO: {stage_name}", flush=True)
    print(f"Script: {script_path}", flush=True)
    print("=" * 72, flush=True)

    # Fuerza UTF-8 en los procesos Python hijos, independientemente
    # de la página de códigos configurada en Windows.
    process_env = os.environ.copy()
    process_env["PYTHONUTF8"] = "1"
    process_env["PYTHONIOENCODING"] = "utf-8"

    process = subprocess.Popen(
        [sys.executable, "-u", str(script_path), *(args or [])],
        cwd=PROJECT_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
        env=process_env,
    )

    assert process.stdout is not None
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
    finally:
        process.stdout.close()

    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, process.args)

    print(f"\\nFIN: {stage_name} (código de salida 0)", flush=True)


## 3. Entrenar y evaluar el modelo

Esta etapa entrena y evalúa el modelo como en el flujo original. Se ejecuta por separado para que el análisis de deriva ocurra antes de generar predicciones. Si el entrenamiento falla, detén el flujo y revisa el error.

In [3]:
run_script(TRAIN_SCRIPT, "ENTRENAMIENTO Y EVALUACIÓN")
print("\nEtapa de entrenamiento finalizada correctamente.")

INICIO: ENTRENAMIENTO Y EVALUACIÓN
Script: C:\Users\angry\source\repos\repos_otros\6.Wizeline_test\DataScientist_wizeline\src\train.py
ENTRENAMIENTO Y EVALUACIÓN DE REGRESIÓN
Filas totales: 800 | desarrollo: 640 | test reservado: 160
Predictores originales: 20 | CV: 5 folds
Correlación absoluta permitida: [0.05, 0.95]
Buscando configuración con menor RMSE medio de validación cruzada...
Procesadores lógicos disponibles: 12
Trabajos de entrenamiento concurrentes: 11
Fitting 5 folds for each of 128 candidates, totalling 640 fits
Tiempo de búsqueda CV: 12.83 segundos (0.21 minutos)

Modelo aceptado y guardado: C:\Users\angry\source\repos\repos_otros\6.Wizeline_test\DataScientist_wizeline\models\best_model.joblib

--- Resultado ---
Modelo seleccionado: XGBRegressor
Mejor RMSE medio CV: 2.205
RMSE desarrollo (80 %): 1.433
RMSE test reservado (20 %): 1.955
MAE test: 1.612 | R² test: 0.830
RMSE entrenamiento CV: 1.438
RMSE validación CV: 2.205
Brecha absoluta CV: 0.766
Brecha absoluta entrenam

## 4. Evaluar la deriva de datos antes de predecir

Este paso compara `training_data.csv` con `blind_test_data.csv` mediante PSI para cada característica. El informe se guarda en `reports/data_drift_psi.csv`.

Los umbrales son provisionales y el resultado es informativo: la deriva no equivale necesariamente a una pérdida de precisión. En esta primera versión, el análisis no bloquea la predicción ni activa un reentrenamiento automático.

In [4]:
import pandas as pd

run_script(DRIFT_SCRIPT, "MONITORIZACIÓN DE DERIVA DE DATOS (PSI)")

if not DRIFT_REPORT_PATH.is_file():
    raise FileNotFoundError(
        f"No se generó el informe de deriva esperado: {DRIFT_REPORT_PATH}"
    )

drift_df = pd.read_csv(DRIFT_REPORT_PATH)
print(f"\nInforme de deriva disponible: {DRIFT_REPORT_PATH}")
display(drift_df)

INICIO: MONITORIZACIÓN DE DERIVA DE DATOS (PSI)
Script: C:\Users\angry\source\repos\repos_otros\6.Wizeline_test\DataScientist_wizeline\src\drift_monitoring.py

Informe de deriva de datos
   feature      psi     status
 feature_0 0.127359 SIN_ALERTA
feature_17 0.112631 SIN_ALERTA
 feature_5 0.098257 SIN_ALERTA
feature_19 0.080407 SIN_ALERTA
 feature_6 0.080386 SIN_ALERTA
feature_14 0.071485 SIN_ALERTA
feature_18 0.058641 SIN_ALERTA
feature_15 0.057663 SIN_ALERTA
 feature_4 0.053039 SIN_ALERTA
feature_11 0.050454 SIN_ALERTA
 feature_2 0.044909 SIN_ALERTA
 feature_7 0.044451 SIN_ALERTA
 feature_3 0.043992 SIN_ALERTA
feature_13 0.041780 SIN_ALERTA
feature_16 0.038258 SIN_ALERTA
 feature_9 0.036281 SIN_ALERTA
feature_12 0.036113 SIN_ALERTA
 feature_8 0.027553 SIN_ALERTA
feature_10 0.022447 SIN_ALERTA
 feature_1 0.009348 SIN_ALERTA

Informe guardado en: C:\Users\angry\source\repos\repos_otros\6.Wizeline_test\DataScientist_wizeline\reports\data_drift_psi.csv

Nota: el PSI detecta cambios de d

,feature,psi,status
0,feature_0,0.127359,SIN_ALERTA
1,feature_17,0.112631,SIN_ALERTA
2,feature_5,0.098257,SIN_ALERTA
3,feature_19,0.080407,SIN_ALERTA
4,feature_6,0.080386,SIN_ALERTA
5,feature_14,0.071485,SIN_ALERTA
6,feature_18,0.058641,SIN_ALERTA
7,feature_15,0.057663,SIN_ALERTA
8,feature_4,0.053039,SIN_ALERTA
9,feature_11,0.050454,SIN_ALERTA


## 5. Generar predicciones sobre el conjunto ciego

Una vez completado el análisis de deriva, se ejecuta la predicción. En esta versión, las alertas PSI se registran para revisión, pero no bloquean automáticamente el proceso.

In [5]:
run_script(PREDICT_SCRIPT, "PREDICCIÓN SOBRE EL CONJUNTO CIEGO")

print("\nFlujo de entrenamiento, monitorización y predicción finalizado.")
print(f"Archivo de predicciones esperado: {PREDICTIONS_PATH}")

INICIO: PREDICCIÓN SOBRE EL CONJUNTO CIEGO
Script: C:\Users\angry\source\repos\repos_otros\6.Wizeline_test\DataScientist_wizeline\src\predict.py
Filas procesadas: 200
Modelo utilizado: C:\Users\angry\source\repos\repos_otros\6.Wizeline_test\DataScientist_wizeline\models\best_model.joblib
Archivo generado: C:\Users\angry\source\repos\repos_otros\6.Wizeline_test\DataScientist_wizeline\outputs\blind_test_predictions.csv
Columna de predicción: target_pred
\nFIN: PREDICCIÓN SOBRE EL CONJUNTO CIEGO (código de salida 0)

Flujo de entrenamiento, monitorización y predicción finalizado.
Archivo de predicciones esperado: C:\Users\angry\source\repos\repos_otros\6.Wizeline_test\DataScientist_wizeline\outputs\blind_test_predictions.csv


## 6. Comprobar el archivo de predicciones

Esta celda comprueba la existencia del CSV, la columna `target_pred`, el número de filas y la validez numérica de las predicciones. No vuelve a entrenar ni modifica el archivo.

In [6]:
import numpy as np
import pandas as pd

if not PREDICTIONS_PATH.is_file():
    raise FileNotFoundError(f"No se generó el archivo esperado: {PREDICTIONS_PATH}")

predictions_df = pd.read_csv(PREDICTIONS_PATH)

if "target_pred" not in predictions_df.columns:
    raise ValueError("El archivo no contiene la columna requerida 'target_pred'.")

predicted_values = pd.to_numeric(predictions_df["target_pred"], errors="coerce")
if predicted_values.isna().any() or not np.isfinite(predicted_values.to_numpy()).all():
    raise ValueError("'target_pred' contiene valores faltantes o no finitos.")

print(f"Archivo: {PREDICTIONS_PATH}")
print(f"Filas: {len(predictions_df)}")
print(f"Columnas totales: {len(predictions_df.columns)}")
print(f"Valores faltantes en target_pred: {predicted_values.isna().sum()}")
print(f"Mínimo de target_pred: {predicted_values.min():.3f}")
print(f"Máximo de target_pred: {predicted_values.max():.3f}")
print(f"Media de target_pred: {predicted_values.mean():.3f}")

display(predictions_df.head())

Archivo: C:\Users\angry\source\repos\repos_otros\6.Wizeline_test\DataScientist_wizeline\outputs\blind_test_predictions.csv
Filas: 200
Columnas totales: 21
Valores faltantes en target_pred: 0
Mínimo de target_pred: 1.973
Máximo de target_pred: 25.188
Media de target_pred: 14.176


,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,...,feature_11,feature_12,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19,target_pred
0,676.867615,32.518822,254.825875,502.268510,609.469688,497.624266,105.246239,269.045539,150.177005,312.649860,...,0.237996,660.030637,147.059794,40.232132,464.424834,121.147466,68.284243,26.969870,314.461582,12.299208
1,628.695228,426.163933,347.070280,431.106903,915.527507,301.699534,1.666992,306.733041,104.234252,63.242070,...,6.608084,600.994184,43.619815,48.153926,457.256565,49.163652,85.511662,33.500538,819.537877,12.642783
2,131.765943,323.839669,245.399775,181.814398,710.179159,59.117377,312.622788,687.965027,109.803179,381.169500,...,1.822370,736.306092,138.759029,36.915389,436.174065,10.037994,62.631938,6.211169,341.361374,14.656399
3,160.970195,489.712029,70.482159,309.486269,888.030604,412.655666,216.124989,47.415477,104.139145,326.462385,...,1.686895,485.144327,143.668518,27.168148,309.715497,149.661493,66.415878,15.001753,539.087409,12.173369
4,419.907137,216.625219,487.887860,253.704462,323.226862,65.744463,271.811469,527.726782,129.805782,168.429679,...,0.948507,365.946758,72.337904,36.232169,302.772338,186.944884,106.514846,3.443809,364.341969,18.646908


## Orden recomendado de ejecución

1. Selecciona como kernel el entorno `.venv` del proyecto.
2. Ejecuta las celdas de arriba hacia abajo.
3. El cuaderno ejecuta entrenamiento y evaluación, después analiza el PSI y finalmente genera las predicciones.
4. Revisa `reports/data_drift_psi.csv` y `outputs/blind_test_predictions.csv`.
5. Para repetir el flujo completo, vuelve a ejecutar las celdas de las etapas 3, 4 y 5 en ese orden.

## 7. Probar drift y reentrenamiento con datos sintéticos

Esta etapa es independiente de la predicción sobre el conjunto ciego. Compara `training_data.csv` con `new_labeled_data.csv` mediante PSI y después evalúa el modelo vigente y tres estrategias de reentrenamiento usando `retraining_evaluation.csv`.

**Importante:** ambos CSV se generaron sintéticamente para probar el código. Los resultados no deben interpretarse como evidencia de drift real ni como una comparación válida del rendimiento esperado en producción. El script guarda los candidatos por separado y no reemplaza automáticamente el modelo aprobado.

Los archivos esperados son `data/new_labeled_data.csv` y `data/retraining_evaluation.csv`.

In [7]:
# Validar los archivos sintéticos necesarios antes de ejecutar el proceso.
for required_path in (RETRAIN_SCRIPT, NEW_LABELED_PATH, RETRAIN_EVALUATION_PATH):
    if not required_path.is_file():
        raise FileNotFoundError(
            f"No se encontró el archivo requerido: {required_path}\n"
            "Comprueba que retrain.py y los dos CSV sintéticos estén en sus rutas esperadas."
        )

run_script(
    RETRAIN_SCRIPT,
    "DRIFT Y COMPARACIÓN DE ESTRATEGIAS DE REENTRENAMIENTO",
    args=[
        "--new-labeled", str(NEW_LABELED_PATH),
        "--evaluation", str(RETRAIN_EVALUATION_PATH),
    ],
)

# Verificar y mostrar ambos informes generados por retrain.py.
for report_path in (RETRAIN_DRIFT_REPORT_PATH, RETRAIN_COMPARISON_PATH):
    if not report_path.is_file():
        raise FileNotFoundError(f"No se generó el informe esperado: {report_path}")

retraining_drift_df = pd.read_csv(RETRAIN_DRIFT_REPORT_PATH)
retraining_comparison_df = pd.read_csv(RETRAIN_COMPARISON_PATH)

print("\nInforme PSI de los datos sintéticos nuevos:")
display(retraining_drift_df)

print("\nComparación del modelo vigente y los candidatos:")
display(retraining_comparison_df)

print("\nNota: el PSI detecta cambios en las entradas; no mide directamente el error predictivo.")
print("La evaluación usa etiquetas sintéticas y sirve únicamente como prueba técnica.")

INICIO: DRIFT Y COMPARACIÓN DE ESTRATEGIAS DE REENTRENAMIENTO
Script: C:\Users\angry\source\repos\repos_otros\6.Wizeline_test\DataScientist_wizeline\src\retrain.py

Analizando drift entre los datos históricos y los nuevos...

Informe de drift:
   feature      psi     status
 feature_5 0.228232 INVESTIGAR
feature_14 0.200853 INVESTIGAR
 feature_7 0.142877 SIN_ALERTA
 feature_0 0.103498 SIN_ALERTA
 feature_4 0.069807 SIN_ALERTA
feature_13 0.058636 SIN_ALERTA
feature_16 0.055068 SIN_ALERTA
 feature_1 0.053839 SIN_ALERTA
feature_11 0.051885 SIN_ALERTA
feature_18 0.050519 SIN_ALERTA
 feature_2 0.050396 SIN_ALERTA
feature_17 0.050270 SIN_ALERTA
feature_19 0.048896 SIN_ALERTA
 feature_3 0.044650 SIN_ALERTA
feature_12 0.040209 SIN_ALERTA
feature_15 0.032617 SIN_ALERTA
 feature_9 0.028894 SIN_ALERTA
feature_10 0.025878 SIN_ALERTA
 feature_8 0.016680 SIN_ALERTA
 feature_6 0.014260 SIN_ALERTA

Informe de drift guardado en: C:\Users\angry\source\repos\repos_otros\6.Wizeline_test\DataScientist_wize

,feature,psi,status
0,feature_5,0.228232,INVESTIGAR
1,feature_14,0.200853,INVESTIGAR
2,feature_7,0.142877,SIN_ALERTA
3,feature_0,0.103498,SIN_ALERTA
4,feature_4,0.069807,SIN_ALERTA
5,feature_13,0.058636,SIN_ALERTA
6,feature_16,0.055068,SIN_ALERTA
7,feature_1,0.053839,SIN_ALERTA
8,feature_11,0.051885,SIN_ALERTA
9,feature_18,0.050519,SIN_ALERTA



Comparación del modelo vigente y los candidatos:


,strategy,rmse,mae,r2,mean_error,candidate_path
0,modelo_vigente,1.739618,1.417634,0.897100,-0.114495,C:\Users\angry\source\repos\repos_otros\6.Wize...
1,historico_mas_nuevos,2.071510,1.642670,0.854091,-0.040532,C:\Users\angry\source\repos\repos_otros\6.Wize...
2,historico_ponderado,2.092414,1.651197,0.851132,-0.036940,C:\Users\angry\source\repos\repos_otros\6.Wize...
3,solo_nuevos_recientes,3.329364,2.631874,0.623097,0.136840,C:\Users\angry\source\repos\repos_otros\6.Wize...



Nota: el PSI detecta cambios en las entradas; no mide directamente el error predictivo.
La evaluación usa etiquetas sintéticas y sirve únicamente como prueba técnica.


## Orden recomendado de ejecución

1. Selecciona como kernel el entorno `.venv` del proyecto.
2. Ejecuta las celdas de arriba hacia abajo para entrenar, monitorizar `blind_test_data.csv` y generar predicciones.
3. Ejecuta la sección 7 para probar el drift y las estrategias de reentrenamiento con los dos CSV sintéticos.
4. Revisa `reports/data_drift_psi.csv`, `reports/retraining_drift_psi.csv`, `reports/retraining_comparison.csv` y `outputs/blind_test_predictions.csv`.
5. La sección 7 guarda candidatos, pero no reemplaza automáticamente `models/best_model.joblib`.